Preprocess the Documents

In [ ]:
import os
import re

def load_stopwords(path='HamshahriData/persian_stopwords.txt'):
    stopwords = set()
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            w = normalize_persian(line.strip()) if line.strip() else ""
            if w:
                stopwords.update(w.split())
    return stopwords


def normalize_persian(text):
    # Normalize common Arabic variants and Persian/Arabic digits.
    text = (text.replace("ي", "ی").replace("ى", "ی").replace("ك", "ک")
                .replace("ة", "ه").replace("ۀ", "ه").replace("ؤ", "و")
                .replace("إ", "ا").replace("أ", "ا").replace("ٱ", "ا"))
    text = text.translate(str.maketrans("٠١٢٣٤٥٦٧٨٩", "۰۱۲۳۴۵۶۷۸۹"))

    # Keep Persian/Arabic-script letters, Persian digits and whitespace.
    # English letters, punctuation and other symbols become token separators.
    text = re.sub(r"[^۰-۹ء-ی\s]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text)

    return text.strip()




def tokenize(text, stopwords=None):
    tokens = text.split()               
    if stopwords:
        tokens = [t for t in tokens if t not in stopwords]
    return tokens


def preprocess_document(text, stopwords):
    text = normalize_persian(text)
    tokens = tokenize(text, stopwords)
    tokens.sort()                       # alphabetical order
    unique_terms = sorted(set(tokens))  # unique sorted list
    return tokens, unique_terms



def load_hamshahri_corpus(base_path):
    all_docs = {}

    # Loop through year folders
    for year in ["2003", "2004", "2005", "2006", "2007"]:
        year_path = os.path.join(base_path, year)
        if not os.path.isdir(year_path):
            continue

        # Loop through files only; ignore nested directories and unrelated files.
        for filename in os.listdir(year_path):
            file_path = os.path.join(year_path, filename)
            if not os.path.isfile(file_path):
                continue

            with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
                text = f.read()

            doc_id = filename[:-4] if filename.endswith(".ham") else filename
            all_docs[doc_id] = text

    if not all_docs:
        raise FileNotFoundError(
            f"No documents found under {base_path!r}. Check the corpus path and folder structure."
        )

    return all_docs


def preprocess_hamshahri(base_path):
    raw_docs = load_hamshahri_corpus(base_path)
    stopwords = load_stopwords()

    all_terms = {}
    all_unique = {}

    for doc_id, text in raw_docs.items():
        tokens, unique_terms = preprocess_document(text, stopwords)
        all_terms[doc_id] = tokens
        all_unique[doc_id] = unique_terms

    return all_terms, all_unique


base_path = "./HamshahriData/HamshahriCorpus"
all_terms, all_unique_terms = preprocess_hamshahri(base_path)

print("Example doc ID:", list(all_terms.keys())[0])
print("Tokens:", all_terms[list(all_terms.keys())[0]][:50])
print("Unique terms:", all_unique_terms[list(all_unique_terms.keys())[0]][:50])




Build Dictionary and Postings List

In [ ]:
import math


def build_vocabulary(all_unique_terms):
    vocab = set()
    for terms in all_unique_terms.values():
        vocab.update(terms)
    return sorted(vocab)



# Postings List
def build_inverted_index(all_unique_terms):
    inverted = {}
    for doc_id, terms in all_unique_terms.items():
        for term in terms:
            if term not in inverted:
                inverted[term] = []
            inverted[term].append(doc_id)
    return inverted


def compute_tf(all_terms):
    tf = {}   # {doc_id: {term: count}}
    for doc_id, tokens in all_terms.items():
        tf[doc_id] = {}
        for t in tokens:
            tf[doc_id][t] = tf[doc_id].get(t, 0) + 1
    return tf


def compute_df(inverted_index):
    df = {term: len(postings) for term, postings in inverted_index.items()}
    return df


def compute_idf(df, total_docs):
    idf = {term: math.log(total_docs / df_val) for term, df_val in df.items()}
    return idf



def build_index_structures(all_terms, all_unique_terms):
    vocab = build_vocabulary(all_unique_terms)
    inverted_index = build_inverted_index(all_unique_terms)
    tf = compute_tf(all_terms)
    df = compute_df(inverted_index)
    total_docs = len(all_terms)
    idf = compute_idf(df, total_docs)

    return vocab, inverted_index, tf, df, idf

vocab, inverted, tf, df, idf = build_index_structures(all_terms, all_unique_terms)
print("Vocabulary size:", len(vocab))
print("First 50 terms:", vocab[:50])
sample_term = vocab[500]
print("Term:", sample_term)
print("Postings list:", inverted[sample_term][:10])
sample_doc = list(tf.keys())[0]
print("Term frequencies in:", sample_doc)
print(tf[sample_doc])
print("DF of sample term:", df[sample_term])
print("IDF of sample term:", idf[sample_term])


Probabilistic Model – Okapi BM25

In [ ]:
import math

class BM25_LongQuery:
    def __init__(self, tf, df, idf, all_terms, k1=1.2, b=0.75, k3=1.5):
        self.tf = tf
        self.df = df
        self.idf = idf
        self.all_terms = all_terms
        self.k1 = k1
        self.b = b
        self.k3 = k3
        
        # Document lengths
        self.doc_len = {doc: len(tokens) for doc, tokens in all_terms.items()}
        self.avgdl = sum(self.doc_len.values()) / len(self.doc_len)

    def preprocess_query(self, query, stopwords=None):
        query = normalize_persian(query)
        tokens = tokenize(query, stopwords)
        
        # Compute tf for terms in the query
        q_tf = {}
        for t in tokens:
            q_tf[t] = q_tf.get(t, 0) + 1
            
        return tokens, q_tf

    def score(self, query_terms, query_tf, doc_id):
        score = 0.0
        doc_length = self.doc_len[doc_id]

        for term, f_tq in query_tf.items():
            if term not in self.tf[doc_id]:
                continue

            f_td = self.tf[doc_id][term]          # document term frequency
            
            idf = self.idf.get(term, 0)

            # Document term frequency component
            doc_factor = ((self.k1 + 1) * f_td) / (
                self.k1 * ((1 - self.b) + self.b * (doc_length / self.avgdl)) + f_td
            )

            # Query term frequency component
            query_factor = ((self.k3 + 1) * f_tq) / (self.k3 + f_tq)

            score += idf * doc_factor * query_factor

        return score

    def rank(self, query, stopwords=None, top_k=10):
        query_terms, q_tf = self.preprocess_query(query, stopwords)

        scores = {}
        for doc_id in self.all_terms:
            scores[doc_id] = self.score(query_terms, q_tf, doc_id)

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]


bm25 = BM25_LongQuery(tf, df, idf, all_terms, k1=1.2, b=0.75, k3=1.5)

query = "اقتصاد ایران"
results = bm25.rank(query, stopwords=load_stopwords(), top_k=10)

for doc_id, score in results:
    print(doc_id, score)


In [ ]:
import math

class LanguageModel:
    def __init__(self, tf, all_terms, smoothing="dirichlet", lam=0.2, mu=2000):
        self.tf = tf
        self.all_terms = all_terms
        self.smoothing = smoothing
        self.lam = lam
        self.mu = mu
        
        # Document lengths
        self.doc_len = {doc: len(tokens) for doc, tokens in all_terms.items()}
        self.avgdl = sum(self.doc_len.values()) / len(self.doc_len)
        
        # Collection length
        self.collection_len = sum(self.doc_len.values())
        
        # Collection frequency
        self.cf = {}
        for tokens in all_terms.values():
            for t in tokens:
                self.cf[t] = self.cf.get(t, 0) + 1

    def preprocess_query(self, query, stopwords=None):
        query = normalize_persian(query)
        tokens = tokenize(query, stopwords)
        q_tf = {}
        for t in tokens:
            q_tf[t] = q_tf.get(t, 0) + 1
        return tokens, q_tf

    # Probability P(t|d)
    def probability(self, term, doc_id):
        f_td = self.tf[doc_id].get(term, 0)
        len_d = self.doc_len[doc_id]
        cf_t = self.cf.get(term, 0)
        
        if self.smoothing == "jelinek":
            # JM smoothing
            p_doc = f_td / len_d if len_d > 0 else 0
            p_col = cf_t / self.collection_len
            return (1 - self.lam) * p_doc + self.lam * p_col
        
        else:  # Dirichlet smoothing
            p_col = cf_t / self.collection_len
            return (f_td + self.mu * p_col) / (len_d + self.mu)


    def score(self, query_terms, q_tf, doc_id):
        score = 0.0
        # Each query term contributes once, weighted by its query frequency.
        for term, query_freq in q_tf.items():
            p = self.probability(term, doc_id)
            if p > 0:
                score += query_freq * math.log(p)
        return score


    def rank(self, query, stopwords=None, top_k=10):
        query_terms, q_tf = self.preprocess_query(query, stopwords)
        scores = {}
        for doc_id in self.all_terms:
            scores[doc_id] = self.score(query_terms, q_tf, doc_id)
        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]


lm = LanguageModel(tf, all_terms, smoothing="dirichlet", mu=2000)
results = lm.rank("اقتصاد ایران", stopwords=load_stopwords(), top_k=10)
for doc, score in lm.rank("نفت ایران", top_k=5):
    print(doc, score)


In [ ]:
import math

class VectorSpaceModel:
    def __init__(self, tf, idf, all_terms):
        self.tf = tf
        self.idf = idf
        self.all_terms = all_terms

        # Build TF-IDF vectors for all documents
        self.doc_vectors = {}      # {doc_id: {term: weight}}
        self.doc_norms = {}        # vector length for cosine
        
        self.build_doc_vectors()


    # Build document TF-IDF vectors
    def build_doc_vectors(self):
        for doc_id, term_freqs in self.tf.items():
            vector = {}
            norm_sq = 0

            for term, f_td in term_freqs.items():
                weight = f_td * self.idf.get(term, 0)   # tf * idf
                vector[term] = weight
                norm_sq += weight * weight

            self.doc_vectors[doc_id] = vector
            self.doc_norms[doc_id] = math.sqrt(norm_sq) if norm_sq > 0 else 0


    def make_query_vector(self, query, stopwords=None):
        query = normalize_persian(query)
        tokens = tokenize(query, stopwords)

        # term frequency in query
        q_tf = {}
        for t in tokens:
            q_tf[t] = q_tf.get(t, 0) + 1

        vector = {}
        norm_sq = 0

        for t, f_tq in q_tf.items():
            idf = self.idf.get(t, 0)
            weight = f_tq * idf
            vector[t] = weight
            norm_sq += weight * weight

        return vector, math.sqrt(norm_sq) if norm_sq > 0 else 0

    # Cosine Similarity between query and document
    def cosine(self, q_vec, q_norm, doc_id):
        doc_vec = self.doc_vectors[doc_id]
        doc_norm = self.doc_norms[doc_id]

        if q_norm == 0 or doc_norm == 0:
            return 0

        # dot product
        dot = 0
        for t, wq in q_vec.items():
            dot += wq * doc_vec.get(t, 0)

        return dot / (q_norm * doc_norm)

    # Rank documents
    def rank(self, query, stopwords=None, top_k=10):
        q_vec, q_norm = self.make_query_vector(query, stopwords)

        scores = {}
        for doc_id in self.all_terms:
            scores[doc_id] = self.cosine(q_vec, q_norm, doc_id)

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        return ranked[:top_k]

vsm = VectorSpaceModel(tf, idf, all_terms)

query = "اقتصاد ایران"
results = vsm.rank(query, stopwords=load_stopwords(), top_k=10)

for doc_id, score in results:
    print(doc_id, score)


### Hybrid ranking: weighted Reciprocal Rank Fusion (RRF)

BM25, language-model log likelihood, and cosine similarity scores have different numeric scales, so adding their raw scores can let one model dominate. This version combines **rank positions** instead: each model contributes `weight / (rrf_k + rank)`. The default weights are 0.5, 0.3, and 0.2 for BM25, LM, and VSM. RRF is a rank-fusion baseline; the hybrid is not guaranteed to outperform every component and should be assessed on the evaluation set.


In [ ]:
class HybridModel:
    """Weighted Reciprocal Rank Fusion (RRF).

    RRF fuses ranks rather than raw scores, because BM25, log language-model
    likelihoods, and cosine similarity have incompatible numeric scales.
    """
    def __init__(self, bm25_model, lm_model, vsm_model,
                 w_bm25=0.5, w_lm=0.3, w_vsm=0.2, rrf_k=60):
        self.bm25 = bm25_model
        self.lm = lm_model
        self.vsm = vsm_model
        weights = [w_bm25, w_lm, w_vsm]
        if any(w < 0 for w in weights) or sum(weights) <= 0:
            raise ValueError("Hybrid weights must be non-negative and sum to a positive value.")
        total = sum(weights)
        self.weights = [w / total for w in weights]
        self.rrf_k = rrf_k
        self.all_docs = list(bm25_model.all_terms.keys())

    def rank(self, query, stopwords=None, top_k=10):
        n_docs = len(self.all_docs)
        rankings = [
            self.bm25.rank(query, stopwords=stopwords, top_k=n_docs),
            self.lm.rank(query, stopwords=stopwords, top_k=n_docs),
            self.vsm.rank(query, stopwords=stopwords, top_k=n_docs),
        ]

        final_scores = {doc_id: 0.0 for doc_id in self.all_docs}
        for weight, ranking in zip(self.weights, rankings):
            for rank_position, (doc_id, _score) in enumerate(ranking, start=1):
                final_scores[doc_id] += weight / (self.rrf_k + rank_position)

        return sorted(final_scores.items(), key=lambda item: item[1], reverse=True)[:top_k]


hybrid = HybridModel(
    bm25, lm, vsm,
    w_bm25=0.5, w_lm=0.3, w_vsm=0.2
)

query = "اقتصاد ایران"
results = hybrid.rank(query, stopwords=load_stopwords(), top_k=10)
for doc, score in results:
    print(doc, score)


In [ ]:
def load_relevance_judgements(path):
    relevant = {}
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 2:
                continue
            qid = int(parts[0])
            doc_id = parts[1]

            if qid not in relevant:
                relevant[qid] = set()
            relevant[qid].add(doc_id)
    return relevant

# Prefer the directory name specified in the assignment; support the old typo as fallback.
candidate_paths = [
    "./HamshahriData/RelevanceAssessment/judgements.txt",
    "./HamshahriData/RelativeAssesemnt/judgements.txt",
]
path = next((p for p in candidate_paths if os.path.isfile(p)), None)
if path is None:
    raise FileNotFoundError(
        "Could not find judgements.txt. Check the RelevanceAssessment folder path."
    )
relevant = load_relevance_judgements(path)
print(f"Loaded relevance judgements for {len(relevant)} queries from {path}")

In [ ]:
def load_queries(path):
    queries = {}

    for filename in os.listdir(path):
        if filename.endswith(".q"):
            qid = int(filename.replace(".q", ""))
            with open(os.path.join(path, filename), "r", encoding="utf-8") as f:
                queries[qid] = f.read().strip()
    return queries
path = "./HamshahriData/Queries"
print(load_queries(path))
queries = load_queries("./HamshahriData/Queries")


### Evaluation

The assignment specifies 20 values of α between 1 and 10, so the evaluation below uses 20 evenly spaced values. For each query, the cutoff is `round(α × number_of_relevant_documents)`, with a minimum of one result. These are macro-averaged Precision, Recall, and F1 across queries.


In [ ]:
def precision(retrieved, relevant):
    retrieved_set = set(retrieved)
    if len(retrieved_set) == 0:
        return 0
    return len(retrieved_set & relevant) / len(retrieved_set)


def recall(retrieved, relevant):
    if len(relevant) == 0:
        return 0
    return len(set(retrieved) & relevant) / len(relevant)


def f1(p, r):
    if p + r == 0:
        return 0
    return 2 * p * r / (p + r)


In [ ]:
def evaluate_model(model, queries, relevant, stopwords, alpha_values):
    results = { "precision": [], "recall": [], "f1": [] }

    for alpha in alpha_values:
        all_p, all_r, all_f1 = [], [], []

        for qid, query_text in queries.items():
            rel_docs = relevant.get(qid, set())
            n = len(rel_docs)
            k = max(1, round(alpha * n))  # requested number of documents to retrieve

            ranked = model.rank(query_text, stopwords, top_k=k)
            retrieved_docs = [doc for doc, score in ranked]

            p = precision(retrieved_docs, rel_docs)
            r = recall(retrieved_docs, rel_docs)
            f = f1(p, r)

            all_p.append(p)
            all_r.append(r)
            all_f1.append(f)

        # avg across all queries
        # Macro-average across queries, including queries with no judgments.
        denom = max(1, len(all_p))
        results["precision"].append(sum(all_p) / denom)
        results["recall"].append(sum(all_r) / denom)
        results["f1"].append(sum(all_f1) / denom)

    return results


In [ ]:
import numpy as np

# The assignment asks for 20 alpha values in [1, 10].
alpha_values = np.linspace(1, 10, 20).tolist()
stopwords = load_stopwords()

# Set up models
bm25 = BM25_LongQuery(tf, df, idf, all_terms)
lm   = LanguageModel(tf, all_terms, smoothing="dirichlet", mu=2000)
vsm  = VectorSpaceModel(tf, idf, all_terms)
hybrid = HybridModel(bm25, lm, vsm, w_bm25=0.5, w_lm=0.3, w_vsm=0.2)

# Run evaluation
bm25_res = evaluate_model(bm25, queries, relevant, stopwords, alpha_values)
lm_res   = evaluate_model(lm, queries, relevant, stopwords, alpha_values)
vsm_res  = evaluate_model(vsm, queries, relevant, stopwords, alpha_values)
hybrid_res = evaluate_model(hybrid, queries, relevant, stopwords, alpha_values)


In [ ]:
import matplotlib.pyplot as plt

def plot_metric(metric_name, bm25_res, lm_res, vsm_res, hybrid_res, alpha_values):
    plt.figure(figsize=(8,5))
    plt.plot(alpha_values, bm25_res[metric_name], label="BM25")
    plt.plot(alpha_values, lm_res[metric_name], label="Language Model")
    plt.plot(alpha_values, vsm_res[metric_name], label="VSM (TF-IDF)")
    plt.plot(alpha_values, hybrid_res[metric_name], label="Hybrid")
    plt.xlabel("α")
    plt.ylabel(metric_name.capitalize())
    plt.title(f"{metric_name.capitalize()} vs α")
    plt.legend()
    plt.grid(True)
    plt.show()

plot_metric("precision", bm25_res, lm_res, vsm_res, hybrid_res, alpha_values)
plot_metric("recall", bm25_res, lm_res, vsm_res, hybrid_res, alpha_values)
plot_metric("f1", bm25_res, lm_res, vsm_res, hybrid_res, alpha_values)
